# Q13 - Are the customer master dates maintained, and should they be used?
**Question (locked in Week 3).** The customer master holds three fields that look like history: `customer_since`, `last_purchase_date` and `total_purchase_value`. Phase 3 retention and churn work would use them for tenure, recency and value. Do they agree with what the order lines show, for all 500 customers, and should they be used?

**Approach.** For every customer, rebuild the same facts from the order lines (first order, last order, revenue) and compare. A field is called usable only if it clears a bar fixed **before** looking at the results: at least 95% of customers within 30 days (or within 10% for value) **and** a rank correlation of at least 0.9 with the order-derived value. Between those and a correlation of 0.5 it is usable with a caveat; below that it is unusable.

**Basis.** Delivered orders (18,033 of 20,000; cancelled orders set aside, as in Weeks 3 and 5), revenue as `line_total` before GST. A sensitivity run on all orders is shown.

**Method.** Share of customers on each side of the order-derived date; Spearman correlation with a permutation test (5,000 shuffles) for independence; Kolmogorov-Smirnov against a uniform spread; Kruskal-Wallis with an effect size by region and by customer type, with Holm adjustment across the six tests. Seed 42.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    # Environment variable first, then this folder and every parent folder, using the repository layout.
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
MERGED = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
CLEAN = find_dir("CADETX_CLEAN", "customers_clean.csv", ["clean", "data/clean", "week-02/reports/week_02/clean", "reports/week_02/clean"])
assert MERGED is not None and CLEAN is not None, "Could not find the Week 2 merged and clean folders. Run from inside the repository, or set CADETX_MERGED and CADETX_CLEAN to the folders."
OUT=Path("q13_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q13_charts"); CH.mkdir(exist_ok=True)
SEED=42; NPERM=5000
rng=np.random.default_rng(SEED)
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
def kw_eps(groups):
    H,p=stats.kruskal(*groups); n=sum(len(g) for g in groups); k=len(groups); return float(H),float(p),float(max(0.0,(H-k+1)/(n-k)))
def perm_rho(x,y,n=NPERM):
    obs=stats.spearmanr(x,y)[0]; xr=pd.Series(x).rank().values; yr=pd.Series(y).rank().values
    d=np.array([np.corrcoef(rng.permutation(xr),yr)[0,1] for _ in range(n)]); return float(obs),float((1+(np.abs(d)>=abs(obs)).sum())/(n+1))
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s_ in ("top","right"): ax.spines[s_].set_visible(False)
    for s_ in ("left","bottom"): ax.spines[s_].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=0.8); ax.set_axisbelow(True)
print("Folders found in:",Path(*MERGED.resolve().parts[-4:]),"and",Path(*CLEAN.resolve().parts[-4:]))

Folders found in: week-02/reports/week_02/merged and week-02/reports/week_02/clean


## Part 0. Load, rebuild the history from the order lines, and check

In [2]:
cust=pd.read_csv(CLEAN/"customers_clean.csv",parse_dates=["customer_since","last_purchase_date"])
s=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"])
check("customers",500,len(cust),len(cust)==500 and cust.customer_id.is_unique)
check("sales lines",130402,len(s),len(s)==130402)
dl=s[s.order_status=="Delivered"]
check("delivered orders",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
def build(df):
    g=df.groupby("customer_id").agg(first_order=("order_date","min"),last_order=("order_date","max"),orders=("so_id","nunique"),revenue=("line_total","sum")).reset_index()
    return g.merge(cust,on="customer_id",how="right")
H=build(dl); A=build(s)
check("every customer has delivered orders",0,int(H.first_order.isna().sum()),int(H.first_order.isna().sum())==0)
check("order-derived orders add up to delivered orders",18033,int(H.orders.sum()),int(H.orders.sum())==18033)
check("order-derived revenue adds up to delivered revenue",int(dl.line_total.sum()),int(H.revenue.sum()),int(H.revenue.sum())==int(dl.line_total.sum()))
# independent recomputation for 50 customers with plain filtering
ids=cust.customer_id.sample(50,random_state=SEED).tolist(); ok=True
for i in ids:
    g=dl[dl.customer_id==i]; r=H[H.customer_id==i].iloc[0]
    ok&=(g.order_date.min()==r.first_order) and (g.order_date.max()==r.last_order) and (g.so_id.nunique()==r.orders) and abs(g.line_total.sum()-r.revenue)<1e-6
check("order-derived history matches plain filtering for 50 customers",True,bool(ok),bool(ok))
H["since_gap"]=(H.customer_since-H.first_order).dt.days; H["last_gap"]=(H.last_purchase_date-H.last_order).dt.days
H["value_ratio"]=H.total_purchase_value/H.revenue
print(f"First orders: {H.first_order.min().date()} to {H.first_order.max().date()} (all in 2019); last orders: {H.last_order.min().date()} to {H.last_order.max().date()} (all in 2024)")
check("every customer's first order is in 2019",500,int((H.first_order.dt.year==2019).sum()),int((H.first_order.dt.year==2019).sum())==500)
check("every customer's last order is in 2024",500,int((H.last_order.dt.year==2024).sum()),int((H.last_order.dt.year==2024).sum())==500)
check("master last_purchase_date is never before customer_since",0,int((H.last_purchase_date<H.customer_since).sum()),int((H.last_purchase_date<H.customer_since).sum())==0)

First orders: 2019-01-01 to 2019-12-18 (all in 2019); last orders: 2024-01-08 to 2024-12-31 (all in 2024)


## Part 1. `customer_since` against the first order
If the field were maintained, no customer could have a `customer_since` **after** the date of their first order, and customers should sit close to their first order date.

In [3]:
sg=H.since_gap; n=len(H)
after=(sg>0).mean()*100; before=(sg<0).mean()*100; within=(sg.abs()<=30).mean()*100
rho1,p1=perm_rho(H.customer_since,H.first_order)
ks1=stats.kstest(((H.customer_since-pd.Timestamp("2015-01-01")).dt.days)/ (pd.Timestamp("2024-12-31")-pd.Timestamp("2015-01-01")).days,"uniform")
sens_after=(A.since_gap if "since_gap" in A else (A.customer_since-A.first_order).dt.days)
sens_after=((A.customer_since-A.first_order).dt.days>0).mean()*100
yr=pd.DataFrame({"year":range(2015,2025)})
yr["master_customer_since"]=yr.year.map(H.customer_since.dt.year.value_counts()).fillna(0).astype(int)
yr["actual_first_order"]=yr.year.map(H.first_order.dt.year.value_counts()).fillna(0).astype(int)
yr["master_last_purchase"]=yr.year.map(H.last_purchase_date.dt.year.value_counts()).fillna(0).astype(int)
yr["actual_last_order"]=yr.year.map(H.last_order.dt.year.value_counts()).fillna(0).astype(int)
yr.to_csv(OUT/"q13_year_comparison.csv",index=False)
H["since_year"]=H.customer_since.dt.year
by_since_year=H.groupby("since_year").agg(customers=("customer_id","count"),median_gap_days=("since_gap","median"),share_since_after_first_order_pct=("since_gap",lambda x:(x>0).mean()*100)).reset_index(); by_since_year.to_csv(OUT/"q13_since_by_year.csv",index=False)
check("shares before and after add up",100.0,round(after+before+(sg==0).mean()*100,6),abs(after+before+(sg==0).mean()*100-100)<1e-6)
for col in ["master_customer_since","actual_first_order","master_last_purchase","actual_last_order"]:
    check(f"year table column {col} covers all 500 customers",500,int(yr[col].sum()),int(yr[col].sum())==500)
check("by-since-year table covers all 500 customers",500,int(by_since_year.customers.sum()),int(by_since_year.customers.sum())==500)
late=H[H.customer_since.dt.year>=2020]
check("every customer with customer_since in 2020 or later had ordered before that date",len(late),int((late.since_gap>0).sum()),int((late.since_gap>0).sum())==len(late))
print(f"customer_since is AFTER the first order for {after:.1f}% of customers (before for {before:.1f}%; within 30 days for {within:.1f}%); on all orders {sens_after:.1f}%")
print(f"Gap in days: median {sg.median():.0f}, 5th to 95th percentile {sg.quantile(.05):.0f} to {sg.quantile(.95):.0f}; skew {stats.skew(sg):+.2f}")
print(f"Spearman with the actual first order: {rho1:+.3f} (permutation p = {p1:.2f})")
print(f"Customers with customer_since in 2020 or later: {len(late)}; all {int((late.since_gap>0).sum())} had already ordered in 2019, before that date")
print(f"customer_since is spread evenly over 2015-2024 (KS against uniform p = {ks1.pvalue:.2f}), while every first order is in 2019")
print(by_since_year.round(1).to_string(index=False))

customer_since is AFTER the first order for 58.4% of customers (before for 41.6%; within 30 days for 1.2%); on all orders 58.6%
Gap in days: median 266, 5th to 95th percentile -1366 to 1931; skew -0.02
Spearman with the actual first order: -0.039 (permutation p = 0.37)
Customers with customer_since in 2020 or later: 242; all 242 had already ordered in 2019, before that date
customer_since is spread evenly over 2015-2024 (KS against uniform p = 0.92), while every first order is in 2019
 since_year  customers  median_gap_days  share_since_after_first_order_pct
       2015         54          -1358.5                                0.0
       2016         44           -936.0                                0.0
       2017         51           -636.0                                0.0
       2018         52           -216.0                                0.0
       2019         57            132.0                               87.7
       2020         39            527.0                     

## Part 2. `last_purchase_date` against the last order
A maintained field would equal the date of the last order (the data ends in December 2024, and every customer ordered in 2024).

In [4]:
lg=H.last_gap
earlier=(lg<0).mean()*100; later=(lg>0).mean()*100; within2=(lg.abs()<=30).mean()*100
rho2,p2=perm_rho(H.last_purchase_date,H.last_order)
ks2=stats.kstest(((H.last_purchase_date-pd.Timestamp("2015-01-01")).dt.days)/(pd.Timestamp("2024-12-31")-pd.Timestamp("2015-01-01")).days,"uniform")
sens_earlier=((A.last_purchase_date-A.last_order).dt.days<0).mean()*100
print(f"last_purchase_date is EARLIER than the last order for {earlier:.1f}% of customers (later for {later:.1f}%, by up to {int(lg.max())} days; within 30 days for {within2:.1f}%); on all orders {sens_earlier:.1f}%")
print(f"Gap in days: median {lg.median():.0f}, 5th to 95th percentile {lg.quantile(.05):.0f} to {lg.quantile(.95):.0f}")
print(f"Spearman with the actual last order: {rho2:+.3f} (permutation p = {p2:.2f})")
print(f"Customers whose master last_purchase_date is in 2024: {int((H.last_purchase_date.dt.year==2024).sum())} of 500; their last order is in 2024: 500 of 500")
check("shares earlier and later add up",100.0,round(earlier+later+(lg==0).mean()*100,6),abs(earlier+later+(lg==0).mean()*100-100)<1e-6)

last_purchase_date is EARLIER than the last order for 93.0% of customers (later for 6.8%, by up to 272 days; within 30 days for 6.6%); on all orders 93.4%
Gap in days: median -675, 5th to 95th percentile -2548 to 17
Spearman with the actual last order: -0.005 (permutation p = 0.92)
Customers whose master last_purchase_date is in 2024: 154 of 500; their last order is in 2024: 500 of 500


## Part 3. `total_purchase_value` against revenue from the order lines

In [5]:
rho3,p3=perm_rho(H.total_purchase_value,H.revenue)
rho3o,_=stats.spearmanr(H.total_purchase_value,H.orders)
r=H.value_ratio; within10=((r>0.9)&(r<1.1)).mean()*100
incl=dl.drop_duplicates("so_id").groupby("customer_id").grand_total.sum().reindex(H.customer_id).values
rho3g=stats.spearmanr(H.total_purchase_value,incl)[0]
print(f"Master total_purchase_value adds up to Rs {H.total_purchase_value.sum()/1e6:,.0f}M against Rs {H.revenue.sum()/1e9:.2f}B of delivered revenue: {H.total_purchase_value.sum()/H.revenue.sum()*100:.1f}%")
print(f"Customers within 10% of their revenue: {within10:.1f}%; ratio quartiles {r.quantile(.25):.3f} / {r.median():.3f} / {r.quantile(.75):.3f}; 95th percentile {r.quantile(.95):.3f}")
print(f"Spearman with revenue: {rho3:+.3f} (permutation p = {p3:.2f}); with order count {rho3o:+.3f}; with revenue including GST {rho3g:+.3f}")
check("no customer is within 10% of their revenue",0.0,round(within10,6),True)

Master total_purchase_value adds up to Rs 1,240M against Rs 22.96B of delivered revenue: 5.4%
Customers within 10% of their revenue: 0.0%; ratio quartiles 0.026 / 0.052 / 0.082; 95th percentile 0.127
Spearman with revenue: -0.091 (permutation p = 0.04); with order count -0.060; with revenue including GST -0.090


## Part 4. Do gaps depend on region or customer type?
Six tests (two gaps and the value ratio, each by region and by type), Holm-adjusted together. Central and North East have 11 customers each, so their group values are shown but are imprecise.

In [6]:
rows=[]; tests=[]
for meas,lab in [("since_gap","customer_since gap (days)"),("last_gap","last_purchase_date gap (days)"),("value_ratio","total_purchase_value / revenue")]:
    for grp in ["region","customer_type"]:
        gs=[g[meas].values for _,g in H.groupby(grp)]; Hs,ps,es=kw_eps(gs); tests.append(dict(measure=lab,grouped_by=grp,groups=len(gs),statistic=Hs,p_value=ps,effect_size_epsilon2=es))
        for k,g in H.groupby(grp): rows.append(dict(measure=lab,grouped_by=grp,group=k,customers=len(g),median=g[meas].median(),mean=g[meas].mean()))
T=pd.DataFrame(tests); T["holm_p"]=holm(T.p_value); T.to_csv(OUT/"q13_group_tests.csv",index=False)
pd.DataFrame(rows).to_csv(OUT/"q13_group_medians.csv",index=False)
check("six group tests run",6,len(T),len(T)==6)
print(T.round(3).to_string(index=False))
print(f"Smallest raw p = {T.p_value.min():.3f}; smallest Holm-adjusted p = {T.holm_p.min():.2f}; largest effect size = {T.effect_size_epsilon2.max():.3f}")

                       measure    grouped_by  groups  statistic  p_value  effect_size_epsilon2  holm_p
     customer_since gap (days)        region       6      8.640    0.124                 0.007   0.746
     customer_since gap (days) customer_type       5      1.552    0.817                 0.000   1.000
 last_purchase_date gap (days)        region       6      5.791    0.327                 0.002   1.000
 last_purchase_date gap (days) customer_type       5      5.103    0.277                 0.002   1.000
total_purchase_value / revenue        region       6      4.824    0.438                 0.000   1.000
total_purchase_value / revenue customer_type       5      1.337    0.855                 0.000   1.000
Smallest raw p = 0.124; smallest Holm-adjusted p = 0.75; largest effect size = 0.007


## Part 5. Verdicts and data-dictionary rule
Bar fixed in advance: **usable** needs at least 95% of customers within tolerance and rank correlation of at least 0.9; **usable with a caveat** needs correlation of at least 0.5; otherwise **unusable**.

In [7]:
def verdict(share,rho): return "Usable" if (share>=95 and rho>=0.9) else ("Usable with a caveat" if rho>=0.5 else "Unusable")
V=pd.DataFrame([
 dict(field="customer_since",compared_with="date of the first order",share_within_tolerance_pct=within,tolerance="30 days",spearman_rho=rho1,permutation_p=p1,key_evidence=f"{after:.1f}% of customers are 'since' AFTER their first order; spread evenly over 2015-2024 while every first order is in 2019"),
 dict(field="last_purchase_date",compared_with="date of the last order",share_within_tolerance_pct=within2,tolerance="30 days",spearman_rho=rho2,permutation_p=p2,key_evidence=f"{earlier:.1f}% are EARLIER than the last order; only {int((H.last_purchase_date.dt.year==2024).sum())} of 500 show 2024 although all 500 ordered in 2024"),
 dict(field="total_purchase_value",compared_with="delivered revenue from the order lines",share_within_tolerance_pct=within10,tolerance="10% of revenue",spearman_rho=rho3,permutation_p=p3,key_evidence=f"adds up to {H.total_purchase_value.sum()/H.revenue.sum()*100:.1f}% of order revenue; no customer within 10%; no rank relationship")])
V["verdict"]=[verdict(a,b) for a,b in zip(V.share_within_tolerance_pct,V.spearman_rho)]
V["rule_for_the_data_dictionary"]=["Do not use for tenure, cohorts or churn. Derive first_order_date from the order lines (delivered orders).","Do not use for recency or churn. Derive last_order_date from the order lines.","Do not use for value or segmentation. Derive customer_revenue and order_count from the order lines."]
V.to_csv(OUT/"q13_field_verdicts.csv",index=False)
check("all three fields tested",3,len(V),len(V)==3)
check("verdicts follow the pre-set bar",True,True,all(v==verdict(a,b) for v,a,b in zip(V.verdict,V.share_within_tolerance_pct,V.spearman_rho)))
print(V[["field","share_within_tolerance_pct","spearman_rho","verdict"]].round(3).to_string(index=False))
H[["customer_id","customer_type","region","first_order","last_order","orders","revenue","customer_since","last_purchase_date","total_purchase_value","since_gap","last_gap","value_ratio"]].rename(columns={"since_gap":"since_gap_days","last_gap":"last_purchase_gap_days","orders":"order_count","revenue":"order_revenue"}).to_csv(OUT/"q13_customer_comparison.csv",index=False)
check("customer comparison table has 500 rows",500,len(H),len(H)==500)
check("six result tables written",6,len(list(OUT.glob("*.csv"))),len(list(OUT.glob("*.csv")))==6)

               field  share_within_tolerance_pct  spearman_rho  verdict
      customer_since                         1.2        -0.039 Unusable
  last_purchase_date                         6.6        -0.005 Unusable
total_purchase_value                         0.0        -0.091 Unusable


## Charts

In [8]:
fig,ax=plt.subplots(figsize=(10,5)); yrs=sg/365.25
ax.hist(yrs,bins=40,color=BLUE,alpha=0.85,edgecolor=SURF); ax.axvline(0,color=ORANGE,lw=2)
top=ax.get_ylim()[1]*1.25; ax.set_ylim(0,top)
ax.text(0.2,top*0.97,f"Right of the line: customer since AFTER first order ({after:.0f}%)",color=INK,fontsize=10,va="top")
ax.text(-0.2,top*0.97,f"Left: before ({before:.0f}%)",color=INK,fontsize=10,va="top",ha="right")
ax.set_xlabel("customer_since minus first order (years)",color=INK2); ax.set_ylabel("Customers",color=INK2)
ax.set_title("customer_since has no link to the first order: it is off by years in both directions",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"q13_01_since_gap.png",dpi=160); plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(13,5)); w=0.4; x=np.arange(10)
for ax,a_,b_,ttl in [(axs[0],"master_customer_since","actual_first_order","Customer since (master) vs first order (orders)"),(axs[1],"master_last_purchase","actual_last_order","Last purchase (master) vs last order (orders)")]:
    ax.bar(x-w/2,yr[a_],width=w,color=ORANGE,label="Master field"); ax.bar(x+w/2,yr[b_],width=w,color=BLUE,label="From the order lines")
    ax.set_xticks(x); ax.set_xticklabels(yr.year,fontsize=9); ax.set_ylabel("Customers",color=INK2); ax.set_title(ttl,color=INK,loc="left",fontsize=11.5); ax.legend(frameon=False,fontsize=9,loc="upper left"); style(ax)
fig.suptitle("Years do not match: all 500 customers first ordered in 2019 and last ordered in 2024",color=INK,x=0.01,ha="left",fontsize=13,fontweight="bold")
fig.tight_layout(rect=(0,0,1,0.94)); fig.savefig(CH/"q13_02_year_mismatch.png",dpi=160); plt.close(fig)

fig,ax=plt.subplots(figsize=(10,5)); lyr=lg/365.25
ax.hist(lyr,bins=40,color=BLUE,alpha=0.85,edgecolor=SURF); ax.axvline(0,color=ORANGE,lw=2)
top=ax.get_ylim()[1]*1.25; ax.set_ylim(0,top)
ax.text(-0.3,top*0.97,f"Left of the line: master date EARLIER than last order ({earlier:.0f}%)",color=INK,fontsize=10,va="top",ha="right")
ax.set_xlabel("last_purchase_date minus last order (years)",color=INK2); ax.set_ylabel("Customers",color=INK2)
ax.set_title("last_purchase_date is almost always years behind the actual last order",color=INK,loc="left",fontsize=12); style(ax)
fig.tight_layout(); fig.savefig(CH/"q13_03_last_gap.png",dpi=160); plt.close(fig)

fig,ax=plt.subplots(figsize=(8,6.2))
ax.scatter(H.revenue/1e6,H.total_purchase_value/1e6,s=14,color=BLUE,alpha=0.55); lim=[min(H.revenue.min(),H.total_purchase_value.min())/1e6,max(H.revenue.max(),H.total_purchase_value.max())/1e6]
ax.plot([10,100],[10,100],color=ORANGE,lw=2); ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(10,110); ax.set_ylim(0.003,150)
ax.text(0.04,0.95,"Orange line: master value = order revenue",transform=ax.transAxes,color=INK,fontsize=10,va="top")
ax.set_xlabel("Revenue from the order lines per customer (Rs million, log scale)",color=INK2); ax.set_ylabel("Master total_purchase_value (Rs million, log scale)",color=INK2)
ax.set_title(f"total_purchase_value is about {H.total_purchase_value.sum()/H.revenue.sum()*100:.0f}% of order revenue",color=INK,loc="left",fontsize=12); style(ax); ax.xaxis.grid(True,color=GRID,lw=0.8)
fig.tight_layout(); fig.savefig(CH/"q13_04_value_vs_revenue.png",dpi=160); plt.close(fig)
check("four charts saved",4,len(list(CH.glob("q13_*.png"))),len(list(CH.glob("q13_*.png")))==4)

## Findings

In [9]:
print(f"1. customer_since: {after:.1f}% of customers are 'customers since' a date after they had already ordered; rho {rho1:+.2f}; verdict {V.verdict[0]}.")
print(f"2. last_purchase_date: {earlier:.1f}% are earlier than the actual last order; only {int((H.last_purchase_date.dt.year==2024).sum())} of 500 show 2024 though all ordered in 2024; rho {rho2:+.2f}; verdict {V.verdict[1]}.")
print(f"3. total_purchase_value: {H.total_purchase_value.sum()/H.revenue.sum()*100:.1f}% of order revenue, none within 10%; rho {rho3:+.2f}; verdict {V.verdict[2]}.")
print(f"4. No pattern by region or customer type (smallest Holm-adjusted p = {T.holm_p.min():.2f}, largest effect size {T.effect_size_epsilon2.max():.3f}): the gaps are not a regional or type-specific data-entry issue.")
print("LIMITS: the order history covers 2019-2024 only, so tenure before 2019 cannot be rebuilt from the orders; delivered orders only (a sensitivity on all orders gives the same picture); the master fields may describe something other than orders (for example, other channels), but nothing in the data says so, and the dataset has no owner to ask.")

1. customer_since: 58.4% of customers are 'customers since' a date after they had already ordered; rho -0.04; verdict Unusable.
2. last_purchase_date: 93.0% are earlier than the actual last order; only 154 of 500 show 2024 though all ordered in 2024; rho -0.00; verdict Unusable.
3. total_purchase_value: 5.4% of order revenue, none within 10%; rho -0.09; verdict Unusable.
4. No pattern by region or customer type (smallest Holm-adjusted p = 0.75, largest effect size 0.007): the gaps are not a regional or type-specific data-entry issue.
LIMITS: the order history covers 2019-2024 only, so tenure before 2019 cannot be rebuilt from the orders; delivered orders only (a sensitivity on all orders gives the same picture); the master fields may describe something other than orders (for example, other channels), but nothing in the data says so, and the dataset has no owner to ask.


## Validation report

In [10]:
rep=pd.DataFrame(checks); rep.insert(0,"Check_ID",range(1,len(rep)+1)); rep.to_csv("validation_report_week06_q13.csv",index=False)
print(rep.Status.value_counts().to_dict()); assert (rep.Status=="PASS").all(); print(f"{len(rep)} checks, all PASS")

{'PASS': 25}
25 checks, all PASS
